In [20]:
# Système
import os
import sys
import time

# Built-in imports
import warnings
from collections import Counter
import pickle

# Data manipulation
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import missingno

# Sklearn imports
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    root_mean_squared_error,
    mean_absolute_error
)

# ML Models - Linear
from sklearn.linear_model import (
    LogisticRegression,
    Perceptron,
    SGDClassifier,
    Lasso,
    LassoCV
)


from sklearn.preprocessing import LabelEncoder

# ML Models - SVM
from sklearn.svm import SVC, LinearSVC

# ML Models - Ensemble
from sklearn.ensemble import RandomForestClassifier

# ML Models - Other
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# Disable warnings
warnings.filterwarnings('ignore')

In [21]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)

# Chargement de la dataset de L1 PCSM
df_L1MPI = pd.read_excel("/content/drive/MyDrive/Memoire/DIORES/Datasets/L1MPI_2018_2019_compiled_data_V3.xlsx", sheet_name='Feuil2', header=0);
df_L1PCSM = pd.read_excel("/content/drive/MyDrive/Memoire/DIORES/Datasets/L1PCSM_2018_2019_compiled_data_V3.xlsx", sheet_name='Feuil2', header=0);
df_L1BCGS = pd.read_excel("/content/drive/MyDrive/Memoire/DIORES/Datasets/L1BCGS_2018_2019_compiled_data_V3.xlsx", sheet_name='Feuil2', header=0);

Mounted at /content/drive/


In [22]:
def preprocess_data(df):
    """
    Effectue le prétraitement des données sur le DataFrame donné.

    Args:
        df (pd.DataFrame): Le DataFrame à prétraiter.

    Returns:
        pd.DataFrame: Le DataFrame prétraité.
    """
    # Remplacer les valeurs manquantes par 0
    df['MENTION'] = df['MENTION'].fillna("NULL")
    df = df.fillna(0)

    # Encodage des séries
    def encode_series(df):
        series = ['S1', 'S2', 'S3']
        for serie in series:
            df[serie] = df['Série'].apply(lambda x: 1 if x == serie else 0)
        # df.drop('Série', axis=1, inplace=True)
        return df

    def label_encode_series(df):
      # Définir un dictionnaire de mapping
      series_mapping = {
          'S1': 0,  # ou 'S1': 0, 'S2': 1, 'S3': 2
          'S2': 1,
          'S3': 2
      }
      # Appliquer le mapping
      df['Série_Encode'] = df['Série'].map(series_mapping)
      return df

    # Encodage du sexe
    def encode_sexe(df):
        df['Homme'] = df['Sexe'].apply(lambda x: 1 if x == 'M' else 0)
        df['Femme'] = df['Sexe'].apply(lambda x: 1 if x == 'F' else 0)
        # df.drop('Sexe', axis=1, inplace=True)
        return df

    def label_encode_sexe(df):
      # Définir un dictionnaire de mapping
      sexe_mapping = {
          'F': 0,  # ou 'F': 0, 'M': 1
          'M': 1
      }
      # Appliquer le mapping
      df['Sexe_Encode'] = df['Sexe'].map(sexe_mapping)
      return df

    def label_encode_RESULTAT(df):
      # Définir un dictionnaire de mapping
      RESULTAT_mapping = {
          'NON ADMIS': 0,  # ou 'NON ADMIS': 0, 'AUTORISE': 1, 'PASSE': 2
          'AUTORISE': 1,
          'PASSE': 2,
      }
      # Appliquer le mapping
      df['RESULTAT_Encode'] = df['RESULTAT'].map(RESULTAT_mapping)
      return df

    def label_encode_SESSION(df):
      # Définir un dictionnaire de mapping
      SESSION_mapping = {
          'Deuxième Session': 0,  # ou 'Deuxième Session': 0, 'Première Session': 1
          'Première Session': 1,
      }
      # Appliquer le mapping
      df['SESSION_Encode'] = df['SESSION'].map(SESSION_mapping)
      return df

    def label_encode_MENTION(df):

      # df['MENTION'] = df['MENTION'].fillna("NULL")

      # Définir un dictionnaire de mapping
      MENTION_mapping = {
          'Passable': 0,  # ou 'Passable': 0, 'Assez-Bien': 1, 'Bien': 2, 'Très-Bien': 3, 'NULL': 4
          'Assez-Bien': 1,
          'Bien': 2,
          'Très-Bien': 3,
          'NULL': 4
      }
      # Appliquer le mapping
      df['MENTION_Encode'] = df['MENTION'].map(MENTION_mapping)
      return df

    # Moyennes par Académie
    def encode_academie_performance(df):
        academie_mean = df.groupby("Académie de l'Ets. Prov.")['Moy. Gle'].mean().to_dict()
        df['Academie perf.'] = df["Académie de l'Ets. Prov."].map(academie_mean)
        # df.drop("Académie de l'Ets. Prov.", axis=1, inplace=True)
        return df

    # Moyennes par Résidence
    def encode_residence_performance(df):
        residence_mean = df.groupby("Résidence")['Moy. Gle'].mean().to_dict()
        df['Residence perf.'] = df["Résidence"].map(residence_mean)
        # df.drop("Résidence", axis=1, inplace=True)
        return df

    # https://www.geeksforgeeks.org/how-to-convert-categorical-string-data-into-numeric-in-python/
    def oneHotEncoding(df, columnName):
        df[columnName] = df[columnName].astype(str)

        le = LabelEncoder()
        label = le.fit_transform(df[columnName])
        # df.drop(columnName, axis=1, inplace=True)
        columnName = columnName + '_Encode'
        df[columnName] = label
        return df

    # Conversion des colonnes non numériques en numériques
    def convert_non_numeric_columns(df):
        non_numeric_cols = df.select_dtypes(include=['object']).columns
        for col in non_numeric_cols:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                pass
        return df

    # Supprimer la colonne "Mention"
    def drop_columns(df, columns):
        df.drop(columns, axis=1, inplace=True)
        return df

    # Appel des sous-fonctions
    df = encode_series(df)
    df = encode_sexe(df)

    df = label_encode_series(df)
    df = label_encode_sexe(df)

    df = label_encode_RESULTAT(df)
    df = label_encode_SESSION(df)
    df = label_encode_MENTION(df)

    df = encode_academie_performance(df)
    df = encode_residence_performance(df)

    df = oneHotEncoding(df, "Résidence")
    df = oneHotEncoding(df, "Ets. de provenance")
    df = oneHotEncoding(df, "Centre d'Ec.")
    df = oneHotEncoding(df, "Académie de l'Ets. Prov.")
    df = oneHotEncoding(df, "REGION_DE_NAISSANCE")

    # df = drop_columns(df, ["Mention"])
    # df = convert_non_numeric_columns(df)

    # df = drop_columns(df, ["Sexe"])
    # df = drop_columns(df, ["Série"])

    return df

In [23]:
df_L1PCSM_processed = preprocess_data(df_L1PCSM)
df_L1MPI_processed = preprocess_data(df_L1MPI)
df_L1BCGS_processed = preprocess_data(df_L1BCGS)

In [24]:
df_L1PCSM_processed

,REGION_DE_NAISSANCE,CREDIT,NIVEAU,SESSION,MENTION,MOYENNE ANNUELLE,RESULTAT,RESULTAT APP EVALUATION,Année BAC,Sexe,...,RESULTAT_Encode,SESSION_Encode,MENTION_Encode,Academie perf.,Residence perf.,Résidence_Encode,Ets. de provenance_Encode,Centre d'Ec._Encode,Académie de l'Ets. Prov._Encode,REGION_DE_NAISSANCE_Encode
0,Diourbel,10,1,Deuxième Session,NULL,4.67,NON ADMIS,1,2018,M,...,0,0,4,10.451842,10.248947,73,199,4,1,1
1,Thiès,40,1,Deuxième Session,NULL,9.99,NON ADMIS,1,2018,M,...,0,0,4,10.549706,10.625600,77,277,20,14,9
2,Dakar,40,1,Deuxième Session,NULL,9.55,NON ADMIS,1,2018,F,...,0,0,4,10.570366,10.612075,17,2,154,0,0
3,Thiès,46,1,Deuxième Session,NULL,9.80,AUTORISE,1,2018,F,...,1,0,4,10.549706,10.443462,142,114,22,14,9
4,Dakar,26,1,Deuxième Session,NULL,8.76,NON ADMIS,1,2018,F,...,0,0,4,10.529098,10.765172,74,55,88,9,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1046,Dakar,60,1,Deuxième Session,Passable,11.30,PASSE,1,2018,F,...,2,0,0,10.529098,10.416290,43,320,173,9,0
1047,Dakar,40,1,Deuxième Session,NULL,10.46,NON ADMIS,1,2018,M,...,0,0,4,10.529098,10.416290,43,242,173,9,0
1048,Kolda,60,1,Première Session,Assez-Bien,12.89,PASSE,1,2018,M,...,2,1,1,10.615862,10.845000,150,131,7,5,4
1049,Fatick,31,1,Deuxième Session,NULL,8.98,NON ADMIS,1,2018,M,...,0,0,4,10.526071,10.175000,21,155,54,2,2


In [25]:
df_L1MPI_processed

,REGION_DE_NAISSANCE,CREDIT,NIVEAU,SESSION,MENTION,MOYENNE ANNUELLE,RESULTAT,RESULTAT APP EVALUATION,Année BAC,Sexe,...,RESULTAT_Encode,SESSION_Encode,MENTION_Encode,Academie perf.,Residence perf.,Résidence_Encode,Ets. de provenance_Encode,Centre d'Ec._Encode,Académie de l'Ets. Prov._Encode,REGION_DE_NAISSANCE_Encode
0,Dakar,3,1,Deuxième Session,NULL,2.50,NON ADMIS,1,2018,F,...,0,0,4,10.693385,10.663333,20,135,97,9,0
1,Dakar,40,1,Deuxième Session,NULL,8.74,NON ADMIS,1,2018,M,...,0,0,4,11.080526,12.524286,57,106,17,0,0
2,Tambacounda,60,1,Deuxième Session,Passable,10.97,PASSE,1,2018,M,...,2,0,0,11.000000,11.202000,73,123,89,13,8
3,Kaolack,47,1,Deuxième Session,NULL,8.34,AUTORISE,1,2018,M,...,1,0,4,10.863333,10.740000,54,118,86,4,3
4,Kaolack,60,1,Deuxième Session,Passable,10.31,PASSE,1,2018,F,...,2,0,0,10.863333,10.180000,49,82,59,4,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343,Kaolack,23,1,Deuxième Session,NULL,7.57,NON ADMIS,1,2018,M,...,0,0,4,10.693385,10.744286,32,70,103,9,3
344,Fatick,8,1,Deuxième Session,NULL,4.36,NON ADMIS,1,2018,M,...,0,0,4,10.715652,11.770000,43,77,55,2,2
345,Diourbel,46,1,Deuxième Session,NULL,9.14,AUTORISE,1,2017,M,...,1,0,4,11.153636,11.460000,12,136,98,1,1
346,Matam,3,1,Deuxième Session,NULL,5.25,NON ADMIS,1,2018,M,...,0,0,4,11.285714,11.820000,41,74,53,8,6


In [26]:
df_L1PCSM_processed.dtypes

,0
REGION_DE_NAISSANCE,object
CREDIT,int64
NIVEAU,int64
SESSION,object
MENTION,object
MOYENNE ANNUELLE,float64
RESULTAT,object
RESULTAT APP EVALUATION,int64
Année BAC,int64
Sexe,object


In [27]:
def split_and_save_data(df, train_size=0.8, random_state=None, train_path=None, test_path=None):
   """
   Divise un DataFrame en train et test sets puis les sauvegarde.

   Args:
       df (pandas.DataFrame): Le DataFrame à diviser
       train_size (float): Proportion des données pour le train set (défaut: 0.8)
       random_state (int): Pour la reproductibilité (défaut: None)
       train_path (str): Chemin pour sauvegarder le train set (défaut: None)
       test_path (str): Chemin pour sauvegarder le test set (défaut: None)

   Returns:
       tuple: (df_train, df_test) Les DataFrames train et test
   """
   # Calculer test_size à partir de train_size
   test_size = 1 - train_size

   # Split du dataset
   df_train, df_test = train_test_split(
       df,
       test_size=test_size,
       random_state=random_state
   )

   # Sauvegarder les fichiers si les chemins sont spécifiés
   if train_path:
       df_train.to_csv(train_path, index=False)
       print(f"Train set sauvegardé dans : {train_path}")

   if test_path:
       df_test.to_csv(test_path, index=False)
       print(f"Test set sauvegardé dans : {test_path}")

   return df_train, df_test

In [28]:
def split_all_documents(data_config):
    for dataset_name, config in data_config.items():
        df = config["df"]
        docs = config["docs"]

        for doc_name, paths in docs.items():
            train_path, test_path = paths
            print(f"Splitting {dataset_name} - {doc_name}")
            split_and_save_data(df=df, train_path=train_path, test_path=test_path)

In [29]:
data_config = {
    "L1MPI": {
        "df": df_L1MPI_processed,
        "docs": {
            "doc1": (
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc1/doc1_df_train.csv",
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc1/doc1_df_test.csv"
            ),
            "doc2": (
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc2/doc2_df_train.csv",
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc2/doc2_df_test.csv"
            ),
            "doc3": (
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc3/doc3_df_train.csv",
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc3/doc3_df_test.csv"
            )
        }
    },
    "L1PCSM": {
        "df": df_L1PCSM_processed,
        "docs": {
            "doc1": (
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc1/doc1_df_train.csv",
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc1/doc1_df_test.csv"
            ),
            "doc2": (
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc2/doc2_df_train.csv",
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc2/doc2_df_test.csv"
            ),
            "doc3": (
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc3/doc3_df_train.csv",
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc3/doc3_df_test.csv"
            )
        }
    },
    "L1BCGS": {
        "df": df_L1BCGS_processed,
        "docs": {
            "doc1": (
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc1/doc1_df_train.csv",
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc1/doc1_df_test.csv"
            ),
            "doc2": (
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc2/doc2_df_train.csv",
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc2/doc2_df_test.csv"
            ),
            "doc3": (
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc3/doc3_df_train.csv",
                "/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc3/doc3_df_test.csv"
            )
        }
    }
}


In [30]:
split_all_documents(data_config)

Splitting L1MPI - doc1
Train set sauvegardé dans : /content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc1/doc1_df_train.csv
Test set sauvegardé dans : /content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc1/doc1_df_test.csv
Splitting L1MPI - doc2
Train set sauvegardé dans : /content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc2/doc2_df_train.csv
Test set sauvegardé dans : /content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc2/doc2_df_test.csv
Splitting L1MPI - doc3
Train set sauvegardé dans : /content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc3/doc3_df_train.csv
Test set sauvegardé dans : /content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc3/doc3_df_test.csv
Splitting L1PCSM - doc1
Train set sauvegardé dans : /content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc1/doc1_df_train.csv
Test set sauvegardé dans : /content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc1/doc1_df_test.csv
Splitting L1PCSM - doc2
Train set sauvegardé dans : /cont

In [31]:
# doc1_train_path, doc1_test_path = '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc1/doc1_df_train.csv', '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc1/doc1_df_test.csv'
# doc2_train_path, doc2_test_path = '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc2/doc2_df_train.csv', '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc2/doc2_df_test.csv'
# doc3_train_path, doc3_test_path = '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc3/doc3_df_train.csv', '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc3/doc3_df_test.csv'


# train_df, test_df = split_and_save_data(
#    df=df_L1MPI,
#    train_path= doc1_train_path,
#    test_path= doc1_test_path
# )

# train_df, test_df = split_and_save_data(
#    df=df_L1MPI,
#    train_path= doc2_train_path,
#    test_path= doc2_test_path
# )

# train_df, test_df = split_and_save_data(
#    df=df_L1MPI,
#    train_path= doc3_train_path,
#    test_path= doc3_test_path
# )

In [32]:
# doc1_train_path, doc1_test_path = '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc1/doc1_df_train.csv', '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc1/doc1_df_test.csv'
# doc2_train_path, doc2_test_path = '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc2/doc2_df_train.csv', '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc2/doc2_df_test.csv'
# doc3_train_path, doc3_test_path = '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc3/doc3_df_train.csv', '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc3/doc3_df_test.csv'


# train_df, test_df = split_and_save_data(
#    df=df_L1PCSM,
#    train_path= doc1_train_path,
#    test_path= doc1_test_path
# )

# train_df, test_df = split_and_save_data(
#    df=df_L1PCSM,
#    train_path= doc2_train_path,
#    test_path= doc2_test_path
# )

# train_df, test_df = split_and_save_data(
#    df=df_L1PCSM,
#    train_path= doc3_train_path,
#    test_path= doc3_test_path
# )

In [33]:
# doc1_train_path, doc1_test_path = '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc1/doc1_df_train.csv', '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc1/doc1_df_test.csv'
# doc2_train_path, doc2_test_path = '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc2/doc2_df_train.csv', '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc2/doc2_df_test.csv'
# doc3_train_path, doc3_test_path = '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc3/doc3_df_train.csv', '/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc3/doc3_df_test.csv'


# train_df, test_df = split_and_save_data(
#    df=df_L1BCGS,
#    train_path= doc1_train_path,
#    test_path= doc1_test_path
# )

# train_df, test_df = split_and_save_data(
#    df=df_L1BCGS,
#    train_path= doc2_train_path,
#    test_path= doc2_test_path
# )

# train_df, test_df = split_and_save_data(
#    df=df_L1BCGS,
#    train_path= doc3_train_path,
#    test_path= doc3_test_path
# )